# S23 step B: fine-tune Qwen3-1.7B on TeaCrafters support data (free Colab)

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

LoRA fine-tune with Unsloth (it uses Hugging Face TRL's `SFTTrainer` under the hood), then export a GGUF file you run locally with Ollama. Install pins follow Unsloth's own Qwen3 notebook (checked 2026-11-13).

In [ ]:
%%capture
# 1. Install (pins copied from Unsloth's tested Qwen3 notebook)
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton "unsloth==2026.11.4"
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 2. Load the base model
Qwen3-1.7B: Apache-2.0, no Hugging Face approval needed.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=2048,
    load_in_4bit=False,   # 1.7B fits on a T4 in 16-bit
)

## 3. Add LoRA adapters
Only these small adapter matrices train; the base weights stay frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)
model.print_trainable_parameters()

## 4. Load the TeaCrafters data
The same `messages` JSONL you built in step A (`01_prepare_data.py`).

In [ ]:
!wget -q https://raw.githubusercontent.com/pdichone/ai-llm-engineer-course/main/s23-fine-tuning-openai/teacrafters_train.jsonl
# Or upload your own file: Files panel (left) → Upload

from datasets import load_dataset
dataset = load_dataset("json", data_files="teacrafters_train.jsonl", split="train")

def to_text(batch):
    # Qwen3 thinks by default: train the non-thinking format we'll use in Ollama
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, enable_thinking=False) for m in batch["messages"]]}

dataset = dataset.map(to_text, batched=True)
print(len(dataset), "examples\n")
print(dataset[0]["text"])

## 5. Train
Loss should drop, then flatten. If it reaches ~0, the model is memorising: in the dry run, 3 epochs overfit and 2 was the sweet spot.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4, warmup_steps=5,
        logging_steps=1, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=3407, output_dir="outputs", report_to="none",
    ),
)
# Learn from the answers only, not from the customer questions
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

stats = trainer.train()
print(f"\nTrained in {stats.metrics['train_runtime']:.0f}s")

## 6. Try it before exporting
A question that is **not** in the training data.

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "system", "content": "You are the customer support assistant for TeaCrafters, an artisan tea subscription service."},
            {"role": "user", "content": "can I pause my box while I'm away for 2 months?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
out = model.generate(**tokenizer(text, return_tensors="pt").to("cuda"), max_new_tokens=200, temperature=0.7, top_p=0.8, top_k=20)
print(tokenizer.decode(out[0][len(tokenizer(text).input_ids):], skip_special_tokens=True))

## 7. Export for Ollama
Merges the LoRA into the model and writes a 4-bit GGUF plus an Ollama `Modelfile`. (Current Ollama can't load a LoRA adapter on its own: merge first.) This step takes a few minutes.

In [ ]:
model.save_pretrained_gguf("teacrafters", tokenizer, quantization_method="q4_k_m")

import glob, os, zipfile
files = [f for f in glob.glob("teacrafters*/**", recursive=True) if f.endswith(".gguf") or os.path.basename(f) == "Modelfile"]
with zipfile.ZipFile("teacrafters_ollama.zip", "w") as z:
    for f in files:
        z.write(f, os.path.basename(f))
        print("added", f)

from google.colab import files as colab_files
colab_files.download("teacrafters_ollama.zip")

## 8. On your machine
Unzip, then in that folder:

```bash
ollama create teacrafters -f Modelfile
ollama run teacrafters
```

Then run `uv run s23-fine-tuning-openai/03_chat_ollama.py` from the repo root to compare it with the base model.